# Hallmark Python Demo: init, info, add, commit, checkout, status, and clone


## Setup


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

from hallmark import Repo
from hallmark.downloader import download_remote_data
from hallmark.error import DestinationExistsError


In [ ]:
%%bash
rm -rf repo1
rm -rf hallmark-demo-clone
mkdir repo1


## 1. Initialize the hallmark repo


In [ ]:
repo1 = Repo.init("repo1")


## 2. Create data files on `main`


In [ ]:
%%bash
pushd repo1
for f in a{0,0.75}_i{0,30,60,90,120}.h5; do echo "$f" > "$f"; done
echo "Files in repo1/:"
ls *.h5
popd


## 3. Add files and commit on `main`


In [ ]:
paraframe = repo1.add("a{a}_i{i}.h5")
paraframe

This paraframe object will also be reflected in the data.tsv file.

In [ ]:
repo1.status()

In [ ]:
repo1.commit("add main dataset")


In [ ]:
repo1.status()

In [ ]:
%%bash
echo "Objects stored after commit:"
find repo1/.hm/objects -type f | wc -l


### Commit log


In [ ]:
print(repo1.log())


## 4. Demo for `repo.checkout('BRANCH')`

We use a temporary branch so the main workflow stays intact.


In [ ]:
# add(".") rebuilds the manifest from files that currently exist

repo1.checkout("experiment")

for path in Path(str(repo1.worktree)).glob("a*.h5"):
    path.unlink()

b_files = [
    f"b{a}_i{i}.h5"
    for a in [0, 0.75]
    for i in [0, 30, 60, 90, 120]
    ]

for name in b_files:
    path = Path(str(repo1.worktree)) / name
    path.write_text(f"{name}\n", encoding="utf-8")

repo1.set_config(fmt="b{a}_i{i}.h5")
repo1.add(".")
repo1.commit("add experiment dataset")

repo1.status()


In [ ]:
# Check back to main to see that the manifest has been correctly rebuilt
repo1.checkout("main")

In [ ]:
# Show available branches
repo1.branches()

In [ ]:
%%bash
echo "Files after checkout back to main:"
ls repo1/*.h5


## 5. Filter Functionality


We can also show that we can filter data from the ParaFrame Object

In [ ]:
# Single Parameter Filter
paraframe(a=0)

In [ ]:
# Or Filter
paraframe(a=0, i=30)

In [ ]:
# And Filter
paraframe(a=0)(i=30)

In [ ]:
# Masking Filter retains Pandas DataFrame functionality
paraframe[(30 <= paraframe.i) & (paraframe.i <= 100)]

## 6. Clone Functionality


In [ ]:
# Demo: clone a remote demo repository
from hallmark.error import DestinationExistsError

try:
    clone = Repo.clone('https://github.com/l6a/hallmark-demo-repo.hm.git', "hallmark-demo-clone", progress=True)
    clone_result = {
        "dot_hallmark_repo": str(clone.dothm.path),
        "hallmark_worktree": str(clone.worktree),
        "branches": clone.branches(),
        "data_rows": len(clone.state.data),
    }
except DestinationExistsError as exc:
    clone_result = str(exc)

clone_result
